# 02 — Deterministic Hodgkin-Huxley

Resting state, step-current action potential, gating dynamics, and
per-channel ionic currents. Both classic and energy-landscape modes.

In [1]:
import sys; sys.path.insert(0, '/workspace')
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans','Arimo','DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
FIG_DIR = '/mnt/results/hh_simulator/figures'
os.makedirs(FIG_DIR, exist_ok=True)
from hh_simulator import (NaChannel, KChannel, LeakChannel, PointCell,
    Simulator, step_pulse, analysis, viz)
t_eval = np.linspace(0, 40, 4001)
I_fn = step_pulse((0,40), 10.0, onset=5, dur=30)

In [2]:
cell = PointCell([NaChannel('classic'), KChannel('classic'), LeakChannel()])
print(f'Resting potential: {cell.resting_potential():.3f} mV')
sim = Simulator(cell)
sol = sim.run((0,40), I_inj=I_fn, mode='deterministic', t_eval=t_eval)
aps = analysis.detect_aps(sol.V, sol.t)
print(f'AP peak={np.max(sol.V):.2f} mV, n_spikes={aps["n_spikes"]}')
viz.plot_voltage(sol, savepath=f'{FIG_DIR}/02_voltage_classic.svg')
plt.show()

Resting potential: -65.000 mV


AP peak=40.27 mV, n_spikes=2


In [3]:
viz.plot_gating(sol, savepath=f'{FIG_DIR}/02_gating_classic.svg')
plt.show()

In [4]:
viz.plot_currents(sol, savepath=f'{FIG_DIR}/02_currents_classic.svg')
plt.show()

In [5]:
cell_e = PointCell([NaChannel('energy'), KChannel('energy'), LeakChannel()])
sol_e = Simulator(cell_e).run((0,40), I_inj=I_fn, mode='deterministic', t_eval=t_eval)
fig, ax = plt.subplots(figsize=(7,3.2))
ax.plot(sol.t, sol.V, 'k-', lw=1.2, label='classic HH')
ax.plot(sol_e.t, sol_e.V, '#E9ED4C', lw=1.2, ls='--', label='Eyring energy')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Voltage (mV)')
ax.set_title('Classic vs energy-landscape mode'); ax.legend(frameon=False)
fig.savefig(f'{FIG_DIR}/02_classic_vs_energy.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/02_classic_vs_energy.png', bbox_inches='tight', dpi=150)
plt.show()